# PySpark dataset setup
Run this notebook first to validate your local environment.


> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [ ]:
import sys
print(sys.version)

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window, types as T
from IPython.display import display

# Run from the extracted project directory in VS Code/Jupyter.
BASE = Path.cwd()
if not (BASE / 'data').exists():
    raise FileNotFoundError('Open the notebook from the pyspark_de_practice directory (data/ required).')
spark = (SparkSession.builder.master('local[*]').appName('PySpark DE Interview Practice')
         .config('spark.sql.session.timeZone','UTC')
         .config('spark.sql.shuffle.partitions','8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

csv_tables = ['customers','products','orders','order_items','transactions','pipeline_logs','cdc_changes']
for name in csv_tables:
    globals()[name] = (spark.read.option('header',True).option('inferSchema',True)
                       .csv(str(BASE / 'data' / (name+'.csv'))))
# Explicit nested JSON input (avoid treating nested properties as strings).
events = spark.read.json(str(BASE / 'data' / 'events.jsonl'))
# Corrupt records are preserved in a string column; explicit schema is important.
malformed_events = (spark.read.schema(T.StructType([
    T.StructField('event_id',T.LongType()),
    T.StructField('event_type',T.StringType()),
    T.StructField('_corrupt_record',T.StringType())
])).option('mode','PERMISSIVE').option('columnNameOfCorruptRecord','_corrupt_record')
.json(str(BASE / 'data' / 'malformed_events.jsonl')))
print('Spark version:',spark.version)
for name in csv_tables+['events']:
    print(f'{name}: {globals()[name].count()} rows')


In [ ]:
orders.printSchema()
orders.show(5, truncate=False)
events.printSchema()


In [ ]:
import csv
import json
from pathlib import Path

# Adjust this path if your data folder is elsewhere
DATA = Path("data")

csv_path = DATA / "customers.csv"
json_path = DATA / "customers.json"
jsonl_path = DATA / "customers.jsonl"

# Load existing customer records
with csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)
    fieldnames = reader.fieldnames
    customers = list(reader)

# Select an existing customer with a valid email.
# Preserve existing blank and whitespace-only test cases.
target = next(
    c for c in customers
    if c.get("email") and c["email"].strip()
)

target_id = target["customer_id"]

# Set email to NULL
target["email"] = ""

# Update CSV
with csv_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(customers)

# Update JSON and JSONL, preserving existing field types
with json_path.open("r", encoding="utf-8") as f:
    customers_json = json.load(f)

for c in customers_json:
    if str(c["customer_id"]) == str(target_id):
        c["email"] = None

with json_path.open("w", encoding="utf-8") as f:
    json.dump(customers_json, f, indent=2, ensure_ascii=False)

with jsonl_path.open("w", encoding="utf-8") as f:
    for c in customers_json:
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

print(f"Updated customer_id: {target_id}")
print("CSV, JSON and JSONL updated successfully.")